# Baseline Experiment - Binary Classification - Llama-3.2-3B

Apply Llama-3.2-3B to classify whether a tweet is relevant to a disater or not, in other word, binary classification.

## A. Setup

In [1]:
%pip install transformers tqdm datasets accelerate huggingface_hub python-dotenv peft trl bitsandbytes evaluate

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv()

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
import evaluate
from datasets import Value

import configuration
from src import setup, data_utils, hf_utils
# from src.models import bert


/home/ubuntu/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from huggingface_hub import login
login(os.getenv("HF_TOKEN"))

base_model = "meta-llama/Llama-3.2-3B"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## B. Data

### B.1. Load sets

In [4]:
data_frac = data_utils.DATA_FRACTION

df_train, df_val, df_test = data_utils.load_datasets()

### B.2. Shrink dataset size for development purpose

In [5]:
# Comment out this cell to use the full dataset. This is just for quick testing.
train_size = 100
val_size = int(train_size * len(df_val) / len(df_train))
test_size = int(train_size * len(df_test) / len(df_train))

df_train = df_train.sample(n=train_size, random_state=setup.RANDOM_SEED)
df_val = df_val.sample(n=val_size, random_state=setup.RANDOM_SEED)
df_test = df_test.sample(n=test_size, random_state=setup.RANDOM_SEED)

In [6]:
# Load data as Hugging Face Datasets
ds_train, ds_val, ds_test = hf_utils.create_datasets(df_train, df_val, df_test)

## C. Tokenization

In [7]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
# Llama 3.2 does not have a native padding token, which is required for sequence classification batching.
# We must set the padding token to the EOS token and apply this to both the tokenizer and the model configuration.
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

### C.1. Quick preview on dataset to find optimal parameter for `tokenizer`

In [8]:
hf_utils.max_length_dist(df_train, 'tweet_text', tokenizer)


90th percentile: 48.49999999999999
95th percentile: 55.249999999999986
99th percentile: 60.650000000000006
Absolute Maximum length: 62


So, set the `max_length` to `64`.

### C.2. Do the Tokenization

In [9]:
def format_dataset(dataset):
    # The HF Trainer API expects a column named 'labels'.
    dataset = dataset.rename_column("informative", "labels")
    dataset = dataset.map(lambda x: {"labels": int(x["labels"])})
    # Ensure labels are int64 for cross-entropy loss.
    dataset = dataset.cast_column("labels", Value("int64"))
    dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return dataset

save_path = Path(f"../tokens/LLama_3_2_3B/{data_frac}")
train_tokenized, val_tokenized, test_tokenized = hf_utils.load_or_tokenize(
    ds_train, ds_val, ds_test, tokenizer, save_path,
    force_retokenize=True,
    format_dataset=format_dataset,
)

Tokenizing datasets...


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Casting the dataset: 100%|██████████| 21/21 [00:00<00:00, 8641.26 examples/s]


Saving tokenized datasets to ../tokens/LLama_3_2_3B/0.1...


Saving the dataset (1/1 shards): 100%|██████████| 21/21 [00:00<00:00, 5779.17 examples/s]


In [10]:
def check_dataset_columns(ds, name):
    required = {"input_ids", "attention_mask", "labels"}
    cols = set(ds.column_names)
    missing = required - cols
    print(f"{name} columns:", sorted(cols))
    if missing:
        raise ValueError(f"{name} missing columns: {sorted(missing)}")

check_dataset_columns(train_tokenized, "train")
check_dataset_columns(val_tokenized, "val")

train columns: ['__index_level_0__', 'attention_mask', 'input_ids', 'labels', 'subset', 'tweet_text']
val columns: ['__index_level_0__', 'attention_mask', 'input_ids', 'labels', 'subset', 'tweet_text']


## B. Fine-tuning Llama 3.2-3B base
### B.1. Model

In [11]:
# Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# model with Classification Head
llama_base = AutoModelForSequenceClassification.from_pretrained(
    base_model, num_labels=2, quantization_config=bnb_config, device_map="auto"
)

# Crucial: Sync the padding token id
llama_base.config.pad_token_id = tokenizer.pad_token_id
llama_base.config.problem_type = "single_label_classification"

Loading weights:   0%|          | 1/254 [00:00<03:50,  1.10it/s]/home/ubuntu/miniconda3/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 254/254 [00:08<00:00, 28.85it/s]
[transformers] LlamaForSequenceClassification LOAD REPORT from: meta-llama/Llama-3.2-3B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### B.2. Configure LoRA (PEFT)

In [12]:
# Prepare model for quantized training
llama_base = prepare_model_for_kbit_training(llama_base)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    modules_to_save=["score"] # Instructs PEFT to fully train the classification head
)

peft_model = get_peft_model(llama_base, lora_config)
peft_model.print_trainable_parameters()

trainable params: 24,320,000 || all params: 3,237,075,968 || trainable%: 0.7513


### B.2. Fine-tune

In [13]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

training_args = TrainingArguments(
    output_dir="./llama-3.2-disaster-classifier",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="paged_adamw_32bit",
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    fp16=True,
    report_to="none"
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [14]:
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/home/ubuntu/miniconda3/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss



Thunder v1.0.0

error: unsupported function cuMemAllocManaged

Unified/managed/pinned memory is not supported on prototyping instances.
Switch to a production mode instance to unblock yourself.

This may cause your application to crash if it does not handle the error gracefully.
(Subsequent calls to cuMemAllocManaged will not repeat this message.)

If you believe this is a bug, please report it in our Discord:
  https://discord.gg/nwuETS9jJK

Error operation not supported at line 670 in file /src/csrc/pythonInterface.cpp


: 

### B.3. Predict on the Test set

In [ ]:
predictions = trainer.predict(test_tokenized)
pred_labels = np.argmax(predictions.predictions, axis=-1)
test_acc = accuracy_metric.compute(predictions=pred_labels, references=test_tokenized["informative"])
test_f1 = f1_metric.compute(predictions=pred_labels, references=test_tokenized["informative"])